In [1]:
all = [var for var in globals() if var[0] != "_"]
for var in all:
    del globals()[var]

In [2]:
import numpy as np
import pandas as pd
import random
import calendar
import matplotlib.pyplot as plt
from scipy.stats import mstats
import statsmodels.api as sm
import seaborn as sns
from statsmodels.graphics.gofplots import qqline
import math
import warnings
warnings.filterwarnings('ignore')

In [3]:
############################################################################################
# Settings for the calculations that can be changed
lags = 16  # months of lagged FC
Minimum_Number_Stocks = 0.2  # Firm characteritics that are avvailable for at least 20% of stocks

Upper_limit = 0.98       # FC upper outliers
Lower_limit = 1-Upper_limit # FC lower outliers

def winsorize_df(df):
    return df.apply(winsorize_series, axis=0)

def winsorize_series(s):
    return mstats.winsorize(s, limits=[1-Upper_limit, Lower_limit])

In [4]:
############################################################################################
# Read Data

# # Colab 사용할 경우 구글드라이버 폴더에 접근하게 함
from google.colab import drive
drive.mount('/gdrive')
df = pd.read_excel("/gdrive//MyDrive/QuantFinance/QuantDataR.xlsx", None, index_col=0 )

# # 자산의 컴퓨터에 저장된 데이터 읽기
# data= 'QuantDataR.xlsx'
# df = pd.read_excel(data, None, index_col=0 )  # 자신의 컴퓨터에 설치된 파이썬을 사용하는 경우. 파이썬 코드와 데이터 화일의 위치가 동일해야함. 아니면 폴더 경로를 명시

print("==========================================================================")
print("Spreadsheet 확인")
df_names = list(df.keys())
print(df_names)


Mounted at /gdrive
Spreadsheet 확인
['D_Market', 'D_RI', 'D_TradingValue', 'D_P', 'M_Market', 'Beta', 'Volatility', 'DY', 'RI', 'TradingValue', 'Cash', 'AccReceivables', 'Inventory', 'CurrentAsset', 'R&D', 'Plants', 'Intangible', 'Total Asset', 'AccPayable', 'CurrentLiability', 'LongTermDebt', 'TotalLiability', 'CommonEquity', 'TotalShareEqu', 'Sales', 'CoGS', 'DepArm', 'OperatingIncome', 'EBIT', 'Interest', 'NetIncome', 'DividendPerShare', 'EPS', 'CommonShares', 'Investment', 'Employee', 'WorkingCap', 'MarketValue']


In [5]:
# 개별주식의 총수익률 (RI) 데이터 읽음
RID = df['RI']
print("--------------------------------------------------------------------------")
print("데이터형식 확인")
print(RID)

# Dates
date = RID.index
print("--------------------------------------------------------------------------")
print("Dates")
print(date)

#firm names using 'D_Price'
D_Price = df['D_P']
firm_names =  D_Price.columns
print("--------------------------------------------------------------------------")
print("FIrm Names")
print(firm_names)

--------------------------------------------------------------------------
데이터형식 확인
            A SELF-ADMINISTERED REIT.TST. - TOT RETURN IND  \
Name                                                         
2006-08-31                                             NaN   
2006-09-29                                             NaN   
2006-10-31                                             NaN   
2006-11-30                                             NaN   
2006-12-29                                             NaN   
...                                                    ...   
2020-05-29                                          171.99   
2020-06-30                                          167.76   
2020-07-31                                          170.41   
2020-08-31                                          173.85   
2020-09-30                                          198.19   

            ABLE C&C - TOT RETURN IND  AEKYUNG IND - TOT RETURN IND  \
Name                                  

In [6]:
############################################################################################
# Calculation of log-returns

R = []
print("Null stocks: ", end ='')
for i in range(0,RID.shape[1]):

    if RID.shape[0] - RID.iloc[:, i ].isnull().sum() < 2:

        print (i, ", ", end='')
        R_temp = pd.Series([np.nan]*(RID.shape[0]-1))
        R.append(R_temp)
        continue

    lag0 = (np.log(RID.iloc[1:, i ])).reset_index(drop =True)
    lag1 = (np.log(RID.iloc[0:-1, i ])).reset_index(drop =True)


    R_temp =  (lag0-lag1)*100
    R.append(R_temp)


R = pd.concat(R,axis= 1)
R.index = date[1:]
R.columns =firm_names

R = R.replace([np.inf, -np.inf], np.nan)

Null stocks: 25 , 747 , 

In [7]:
############################################################################################
# Firm Characteristics : Among 37 sheets we need 6th~37th sheet

BE = df["CommonEquity"].iloc[1:,:]
BE.columns = firm_names
FC_date = date

BE_L = BE.shift(lags)   # 16달 lag

ME = (df["MarketValue"] *1000).iloc[1:,:]
ME.columns = firm_names
ME_L = ME.shift(1)       # 1달 lag

FC = (BE_L/ME)

FC = FC.replace([np.inf, -np.inf], np.nan)

In [8]:
# Winsorizing FC (to control outliers)

FC = winsorize_df(FC)

# Nomalization
NFC = (FC -np.mean(FC))/np.std(FC)

# # when you want to pass the processing of winszerizing and nomaization
# NFC = FC

# Replacing NA  to 0
NFC = NFC.fillna(0)

# Check the Winsorized and Normalized FC
NFC.describe()


KeyError: 'In'

,A SELF-ADMINISTERED REIT.TST.,ABLE C&C,AEKYUNG IND,AIR BUSAN,AJ NETWORKS,AJU CAPITAL,AK HOLDINGS,AK PETROCHEMICAL,ALUKO,AMOREPACIFIC,...,YOUNGONE,YOUNGONE HOLDINGS,YOUNGWIRE,YUANTA SECURITIES KOREA,YUHAN,YUHWA SECURITIES,YUNGJIN PHARM,YUYANG D&U SUSP - SUSP.20/03/20,YUYU PHARMA,ZINUS
count,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,...,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000
mean,-0.583807,-1.427983,-0.998970,-0.820579,-0.150779,0.936750,-0.039811,-0.636142,-0.954952,-9.153965,...,-1.478749,-0.712138,0.046199,0.284585,-6.031202,2.791327,-6.696811,-0.867336,0.165907,-0.956361
std,0.912155,1.060767,2.157178,2.157993,0.639153,1.011788,0.954401,0.946405,1.003318,3.118615,...,1.232212,0.981868,0.863020,0.958767,2.176539,1.315432,2.372455,0.994913,0.955829,3.479800
min,-3.040734,-2.728099,-6.566422,-7.165294,-1.803166,-0.450045,-1.160938,-2.356310,-2.447411,-11.553372,...,-3.240790,-1.731238,-1.646968,-1.385207,-8.275337,0.000000,-8.836898,-2.288687,-1.416834,-14.400480
25%,-0.896909,-2.170275,0.000000,0.000000,-0.081664,0.000000,-0.764517,-1.612596,-1.639456,-10.712452,...,-2.779003,-1.476263,-0.360969,-0.371463,-7.265336,2.269968,-8.402492,-1.844741,-0.628964,0.000000
50%,-0.138723,-1.876201,0.000000,0.000000,0.000000,0.793875,-0.341326,0.000000,-0.910911,-10.284257,...,-1.564426,-0.991003,0.000000,0.059247,-6.506078,2.906530,-7.118578,-1.003101,0.000000,0.000000
75%,0.000000,-0.611073,0.000000,0.000000,0.000000,1.813774,0.276627,0.000000,-0.406968,-8.915870,...,0.000000,-0.284680,0.555930,0.999234,-5.978688,3.754749,-6.372529,0.000000,0.956409,0.000000
max,1.283533,2.426405,0.000000,0.000000,2.213324,3.561276,3.011402,1.543131,3.383598,0.000000,...,1.251963,3.026518,3.236884,2.312713,0.000000,5.075321,0.000000,2.488003,2.385744,0.000000


In [9]:
len(NFC.T)

792

In [10]:
results = pd.DataFrame(np.empty((NFC.shape[0],3))*np.nan, index=R.index, columns=['Alpha','Beta','R2'])
Alpha = []
Beta = []
R2 = []


for t in range(lags+1,NFC.shape[0]):

    if NFC.shape[1]-(NFC.iloc[t-1,:]==0).sum()>100:
        XX = sm.add_constant(NFC.iloc[t-1,:]) #### warrning

        #There are object columns in df ,so convert these to float
        XX = pd.DataFrame(XX, dtype='float')

        Y = R.iloc[t,:] # returns of individual stocks at time t

        # remove omit value
        sdata = pd.concat([XX,Y],axis = 1 )
        sdata = sdata.dropna(axis=0).reset_index(drop =True)
        sdata.columns = ['Const','BM','RET']

        XX = sdata[['Const','BM']]

        Y = sdata.loc[:,'RET']

        coef = np.linalg.inv(XX.T @ XX) @ XX.T @ Y

        Alpha.append(coef[0])
        Beta.append(coef[1])

        resid  = Y - XX @ coef.values
        r2 =  1- (resid.T @ resid)  / (sdata.shape[0]-2) / np.var(Y)
        R2.append(r2)

        results.iloc[t,:] = [coef[0], coef[1], r2]

# results = pd.concat([pd.Series(Alpha),
                #    pd.Series(Beta),
                #    pd.Series(R2)], axis=1)

results

,Alpha,Beta,R2
Name,,,
2006-09-29,NaN,NaN,NaN
2006-10-31,NaN,NaN,NaN
2006-11-30,NaN,NaN,NaN
2006-12-29,NaN,NaN,NaN
2007-01-31,NaN,NaN,NaN
...,...,...,...
2020-05-29,4.858583,-0.108126,-0.001151
2020-06-30,-1.506593,-0.028994,-0.002236
2020-07-31,6.718917,-0.007210,-0.002544


In [11]:
print("Alpha")
print("Mean:",results["Alpha"].dropna().mean(), "STD_M:",results["Alpha"].dropna().std() /math.sqrt(results["Alpha"].dropna().shape[0]))
print("Beta")
print("Mean:",results["Beta"].dropna().mean(), "STD_M:",results["Beta"].dropna().std() /math.sqrt(results["Beta"].dropna().shape[0]))
print("R2: ", results["R2"].dropna().mean())


Alpha
Mean: 0.11621387529903955 STD_M: 0.503620254446111
Beta
Mean: 0.026957914689948324 STD_M: 0.02257415240571491
R2:  0.003195476707454998


In [12]:
########################################################################
# Identification of Factors
#######################################################################

RHPS = pd.DataFrame(np.empty((NFC.shape[0],11))*np.nan, index=R.index, columns=['Number_Stocks','High_BM','2','3','Low_BM','RET_High_BM','2','3','4','RET_Low_BM','HML'])

quantile_FC =  NFC.quantile(q=[.2, .4, .6 , .8 ], interpolation='nearest', axis=1).T

for t in range(lags+1, NFC.shape[0]-1):

        if (NFC.shape[1]- NFC.iloc[t,:].isnull().sum())>100:

            RHPS.iloc[t,0] = (NFC.shape[1]- NFC.iloc[t,:].isnull().sum())  # the number of stocks that are used to form portfolios
            Y = R.iloc[t+1,:]
            RHPS.iloc[t,1:5] = quantile_FC.iloc[t,:]

            L = NFC.iloc[t,:] <= quantile_FC.iloc[t,0]
            H = NFC.iloc[t,:] >= quantile_FC.iloc[t,3]
            RHPS.iloc[t+1,10] = Y[H].dropna().mean()  - Y[L].dropna().mean()

            RHPS.iloc[t,1] = quantile_FC.iloc[t,3]
            RHPS.iloc[t,2] = quantile_FC.iloc[t,2]
            RHPS.iloc[t,3] = quantile_FC.iloc[t,1]
            RHPS.iloc[t,4] = quantile_FC.iloc[t,0]

            P1 = (NFC.iloc[t,:] >= quantile_FC.iloc[t,3])
            P2 = (NFC.iloc[t,:] > quantile_FC.iloc[t,2]) & (NFC.iloc[t,:] <= quantile_FC.iloc[t,3])
            P3 = (NFC.iloc[t,:] > quantile_FC.iloc[t,1]) & (NFC.iloc[t,:] <= quantile_FC.iloc[t,2])
            P4 = (NFC.iloc[t,:] > quantile_FC.iloc[t,0]) & (NFC.iloc[t,:] <= quantile_FC.iloc[t,1])
            P5 = (NFC.iloc[t,:] <= quantile_FC.iloc[t,0])


            RHPS.iloc[t+1,5] =  Y[P1].dropna().mean()
            RHPS.iloc[t+1,6] =  Y[P2].dropna().mean()
            RHPS.iloc[t+1,7] =  Y[P3].dropna().mean()
            RHPS.iloc[t+1,8] =  Y[P4].dropna().mean()
            RHPS.iloc[t+1,9] =  Y[P5].dropna().mean()

RHPS

,Number_Stocks,High_BM,2,3,Low_BM,RET_High_BM,2,3,4,RET_Low_BM,HML
Name,,,,,,,,,,,
2006-09-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2020-05-29,792.0,2.035689,0.800885,-0.387838,-2.032101,2.770102,6.049398,5.256834,5.235158,5.255745,-2.485643
2020-06-30,792.0,2.221818,0.912754,-0.338648,-2.052663,-2.301619,-2.924268,-1.377329,-1.749574,0.929426,-3.231045
2020-07-31,792.0,1.933109,0.681485,-0.471411,-2.194796,6.969850,6.412351,6.198079,6.697713,7.275022,-0.305172


In [13]:
Mean = RHPS["2008-03":"2020-07"].mean(axis=0 )
STD_M = (RHPS["2008-03":"2020-07"].std() /math.sqrt(RHPS["2008-03":"2020-07"].shape[0]-lags-1))
t_statistic = Mean/STD_M

result = pd.concat([Mean,STD_M,t_statistic], axis=1)
result.columns = ['Mean','STD_M','t_statistic']
result

,Mean,STD_M,t_statistic
Number_Stocks,792.000000,0.000000,inf
High_BM,0.816110,0.043729,18.662792
2,0.048167,0.020345,2.367544
3,-0.531907,0.023613,-22.526031
Low_BM,-1.740908,0.028691,-60.677362
RET_High_BM,0.716873,0.560978,1.277898
2,0.042488,0.572704,0.074188
3,-0.327140,0.576501,-0.567457
4,-0.299662,0.563046,-0.532217
RET_Low_BM,-0.186206,0.490716,-0.379457


In [14]:
# Read Data for the Market Return and Risk-free rate

Mdata = pd.read_csv("/gdrive//MyDrive/QuantFinance/data.csv",index_col=0 )
Mdata.index = pd.to_datetime(Mdata.index)
Mdata = Mdata.dropna(axis= 0)
n = Mdata.shape[0]
MR = Mdata.loc["2008-03":"2020-07","RKOSPI"]
RF = Mdata.loc["2008-03":"2020-07","RF_M"]


Y = RHPS.loc["2008-03":"2020-07",'HML']
X=  pd.concat([MR-RF],axis = 1)
model = sm.OLS(Y,sm.add_constant(X))
result = model.fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:                    HML   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     6.073
Date:                Thu, 11 Sep 2025   Prob (F-statistic):             0.0149
Time:                        09:37:01   Log-Likelihood:                -376.43
No. Observations:                 149   AIC:                             756.9
Df Residuals:                     147   BIC:                             762.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9028      0.250      3.617      0.0